In [40]:
import importlib
utils = importlib.import_module(".utils", "abt-question-ocr.codes")
import pandas as pd
import re

In [41]:
dir_data = utils.Config.DIR_RESULTS / "amazontextract"
questions_combined = pd.DataFrame()
answers_combined = pd.DataFrame()
for f in dir_data.glob('*.csv'):
    print(f.name)
    if f.name.endswith('_answers.csv'):
        answers = pd.read_csv(f, index_col=0)
        answers['Answer Source'] = f.name.replace('_answers.csv', '.pdf')
        answers_combined = pd.concat([answers_combined, answers])
    elif f.name.endswith('_answers_table.csv'):
        answers = pd.read_csv(f, index_col=0)
        answers['Answers'] = answers[['Guess', 'Actual']].apply(lambda x: x['Actual'] if not pd.isna(x['Actual']) else x['Guess'], axis=1)
        answers['Answer Source'] = f.name.replace('_answers_table.csv', '.pdf')
        answers = answers.drop(columns=['Guess', 'Actual'])
        answers = answers[~pd.isna(answers['Answers'])]
        answers_combined = pd.concat([answers_combined, answers])
    else:
        questions = pd.read_csv(f, index_col=0)
        questions['Question Source'] = f.name.replace('.csv', '.pdf')
        questions_combined = pd.concat([questions_combined, questions])

ABT #17_answers.csv
ABT 2006 recert #23.csv
24-2007-recert no answers.csv
ABT #17.csv
ABT RE No 22 2005.csv
recert 2002- MH group answers_answers_table.csv
ABT2002#19.csv
Document3.csv
ABT #18_answers.csv
Document2.csv
Document6.csv
Document7.csv
ABTexamMay52004 #21.csv
Document5.csv
Document4.csv
Document8.csv
recert18- 2001-MH answers_answers_table.csv
ABT2003#20.csv
ABT2001#18.csv
ABT #19.csv
ABT #18.csv
Document2_answers_table.csv
Document.csv
Document_answers_table.csv
Document7_answers_table.csv
ABT RE No 22 2005 missing page.csv
ABT2000#17.csv
ABT #20.csv


In [42]:
answers_combined[answers_combined['Exam'] == 'Exam booklet page no. 2001'] = 'Exam booklet page no. 2001A'

In [43]:
def mapVal(x):
    grps = re.findall(r'(Recertification(\ Exam)?\ (\d{4})\ Part\ ([A-Z]).*)|(Exam\ booklet\ page\ no\.\ (\d{4}[A-Z]*))', x)
    if len(grps) == 0 or len(grps[0]) != 6: return x
    if grps[0][0] != '':
        return ''.join(grps[0][2:4])
    return grps[0][5]

In [44]:
questions_combined['Exam'] = questions_combined['Exam'].map(mapVal)
answers_combined['Exam'] = answers_combined['Exam'].map(mapVal)

In [45]:
answers_combined = pd.concat([answers_combined, pd.read_csv(utils.Config.DIR_RESULTS / 'combined_answers_from_nonpdfs.csv', index_col=0)]).reset_index(drop=True)
answers_combined

,Questions,Exam,Answers,Answer Source
0,1,2000A,C,ABT #17.pdf
1,2,2000A,D,ABT #17.pdf
2,3,2000A,B,ABT #17.pdf
3,4,2000A,B,ABT #17.pdf
4,5,2000A,D,ABT #17.pdf
...,...,...,...,...
1519,36,2007C,E,ABT Recert 24-2007 MH group Answers.doc
1520,37,2007C,"A, E",ABT Recert 24-2007 MH group Answers.doc
1521,38,2007C,C,ABT Recert 24-2007 MH group Answers.doc
1522,39,2007C,C,ABT Recert 24-2007 MH group Answers.doc


In [46]:
def extractIndex(x):
    res = re.findall(r'([0-9]{1,2})\..*', x)
    if len(res): return res[0]
    return '?'

In [47]:
questions_combined['Question Index'] = questions_combined[['Exam', 'Questions']].apply(lambda x: f"{x['Exam']} Q{extractIndex(x['Questions'])}", axis=1)
answers_combined['Question Index'] = answers_combined['Exam'] + ' Q' + answers_combined['Questions'].astype(str)

In [60]:
questions_combined.query('Exam == "2000A"')[:50]

,Exam,Questions,Options,Question Source,Question Index
0,2000A,5. Contact urticaria is characteristic of,"A) mistletoe, B) crocus, C) pokeweed, D) nettl...",ABT #17.pdf,2000A Q5
1,2000A,6. The dominant route of excretion for absorbe...,"A) bile, B) feces, C) urine, D) epithelial tis...",ABT #17.pdf,2000A Q6
2,2000A,7. Bagassosis is a term for allergic pulmonary...,"A) cotton dust, B) coal dust, C) beryllium, D)...",ABT #17.pdf,2000A Q7
3,2000A,"8. During tidal breathing, inhalation of a mon...","A) bronchus, B) trachea, C) nasopharyngeal reg...",ABT #17.pdf,2000A Q8
4,2000A,9. Which of the following is NOT characteristi...,"A) disorientation, ataxia, depression, B) dila...",ABT #17.pdf,2000A Q9
5,2000A,10. Ingestion of which of the following radioa...,"A) beta emitters, B) alpha emitters, C) gamma ...",ABT #17.pdf,2000A Q10
6,2000A,11. Which of the following chemicals is NOT an...,"A) vitamin E, B) ethoxyquin, C) riboflavin, D)...",ABT #17.pdf,2000A Q11
7,2000A,12. Intranuclear inclusions in the renal tubul...,"A) cobalt, B) mercury, C) antimony, D) lead, E...",ABT #17.pdf,2000A Q12
8,2000A,"13. In some species, accumulation of neurofibr...","A) lead, B) manganese, C) methyl mercury, D) a...",ABT #17.pdf,2000A Q13
9,2000A,14. In the treatment of cyanide (CN) poisoning...,"A) displacing CN from cytochrome oxidase, B) o...",ABT #17.pdf,2000A Q14


In [62]:
def multipleSelectionCriteria(answer):
    match answer:
        case 'A': return '1, 2, 3'
        case 'B': return '1, 3'
        case 'C': return '2, 4'
        case 'D': return '4'
        case 'E': return '1, 2, 3, 4'
        case _: return answer

combined_qa = pd.merge(questions_combined[['Question Index', 'Questions', 'Options', 'Question Source']], answers_combined[['Answers', 'Question Index', 'Answer Source']], on='Question Index', how='inner')
combined_qa = combined_qa.drop_duplicates()
combined_qa['Answers'] = combined_qa.apply(lambda x: multipleSelectionCriteria(x['Answers']) if re.match(r'^\d\)', str(x['Options'])) else x['Answers'], axis=1)

def format(x):
    question = '. '.join(x.iloc[0]['Questions'].split('. ')[1:]).strip()
    if pd.isna(x.iloc[0]['Options']): 
        options = {}
    else:
        try:
            options = {y.split(')')[0].replace('(', "").strip(): ')'.join(y.split(')')[1:]).strip() for y in x.iloc[0]['Options'].split(', ')}
        except:
            print(x)
        
    q_src = ', '.join(x['Question Source'].unique())
    answer = sorted(set([b.strip() for a in x['Answers'].apply(lambda x: x.split(',')).values for b in a]))
    a_src = ', '.join(x['Answer Source'].unique())
    return pd.Series({'Questions': question, 'Options': options, 'Question Source': q_src, 'Answers': answer, 'Answer Source': a_src})

combined_qa = combined_qa.groupby('Question Index')[['Questions', 'Options', 'Question Source', 'Answers', 'Answer Source']].apply(lambda x: format(x)).reset_index()

def sortQA(x):
    df = x.str.split(' Q', expand=True)
    return df.apply(lambda x: [x[0], int(x[1])], axis=1)

combined_qa = combined_qa.sort_values(by='Question Index', key=sortQA)
combined_qa.reset_index(drop=True).to_csv(utils.Config.DIR_RESULTS / "combined_qa.csv")
combined_qa.reset_index(drop=True).to_json(utils.Config.DIR_RESULTS / "combined_qa.json")